### Ablation Studies

In [3]:
from pathlib import Path
from datetime import datetime
import re
import json
import math
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from PIL import Image, ImageOps
from scipy.io import loadmat
from scipy import ndimage

import sys
from pathlib import Path

scripts_path = Path("./scripts").resolve()
src_path = Path("./src").resolve()

for folder_path in [scripts_path, src_path]:
    folder_path = str(folder_path)

    if folder_path not in sys.path:
        sys.path.append(folder_path)

MANUAL_PROJECT_DIR = '.'

PROJECT_DIR = Path(MANUAL_PROJECT_DIR)
# print('PROJECT_DIR =', PROJECT_DIR)

TRACK_IDS = [8, 10, 14, 21]

COMMON_X_START_MM = 20.0
COMMON_X_END_MM = 100.0
COMMON_LENGTH_MM = COMMON_X_END_MM - COMMON_X_START_MM

THERMAL_FPS = 50.0
SCAN_SPEED_MM_PER_S = 10.0
THERMAL_MM_PER_FRAME = SCAN_SPEED_MM_PER_S / THERMAL_FPS
EXTRACTED_THERMAL_FRAMES = int(round(COMMON_LENGTH_MM / THERMAL_MM_PER_FRAME))

SEM_TILE_WIDTH_MM = 6.41

RAW_DIR = PROJECT_DIR / 'data' / 'raw'
THERMAL_DIR = RAW_DIR / 'thermal'
SEM_DIR = RAW_DIR / 'sem'
HEIGHT_DIR = RAW_DIR / 'height_maps'

In [5]:
from feature_extraction import *
train_df = build_feature_dataset(
    track_ids=["Track_8", "Track_10", "Track_14"], n_per_track=300,
    SEM_DIR=SEM_DIR, SEM_TILE_WIDTH_MM=SEM_TILE_WIDTH_MM, seed=0,
)
train_df[["track_id", "x_mm", "cooling_rate_proxy", "thermal_gradient_proxy", "linear_energy_density"]].head(10)

C:\Users\rolabiyi\OneDrive - Arizona State University\Desktop\OneDrive - Arizona State University\Remote PC\NSF Project\NSF-Future-Manufacturing-Challenge\scripts\frame_extraction.py:781: RuntimeWarning: Mean of empty slice
  mean_width = np.nanmean(widths)
C:\Users\rolabiyi\.conda\envs\pd_env\lib\site-packages\numpy\lib\nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\rolabiyi\OneDrive - Arizona State University\Desktop\OneDrive - Arizona State University\Remote PC\NSF Project\NSF-Future-Manufacturing-Challenge\scripts\sem_multimodal_bundle.py:239: RuntimeWarning: Mean of empty slice
  "y_left_mm": float(np.nanmean(dist["y_left_mm"])),
C:\Users\rolabiyi\OneDrive - Arizona State University\Desktop\OneDrive - Arizona State University\Remote PC\NSF Project\NSF-Future-Manufacturing-Challenge\scripts\sem_multimodal_bundle.py:240: RuntimeWarning: Mean of empty slice
  "y_right_mm": float(np.nanmea

,track_id,x_mm,cooling_rate_proxy,thermal_gradient_proxy,linear_energy_density
0,Track_8,70.824884,711.160889,885.217590,20.0
1,Track_8,42.025307,-106.679687,804.520508,20.0
2,Track_8,24.078214,178.250732,744.244812,20.0
3,Track_8,22.160787,-490.101318,828.631409,20.0
4,Track_8,84.653745,633.865967,751.618591,20.0
5,Track_8,92.456934,634.450684,895.967712,20.0
6,Track_8,68.446254,-399.317627,854.675232,20.0
7,Track_8,78.082909,1580.629883,1008.034729,20.0
8,Track_8,63.503967,-785.989990,1038.605103,20.0
9,Track_8,94.207369,-572.486572,667.075623,20.0


In [14]:
test_df = build_feature_dataset(
    track_ids=["Track_21"], n_per_track=300,
    SEM_DIR=SEM_DIR, SEM_TILE_WIDTH_MM=SEM_TILE_WIDTH_MM, seed=42,
)

C:\Users\rolabiyi\OneDrive - Arizona State University\Desktop\OneDrive - Arizona State University\Remote PC\NSF Project\nsf-fmrg-data-challenge-main\notebooks\Extraction Ready\frame_extraction.py:755: RuntimeWarning: Mean of empty slice
  mean_width = np.nanmean(widths)
C:\Users\rolabiyi\.conda\envs\pd_env\lib\site-packages\numpy\lib\nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\rolabiyi\OneDrive - Arizona State University\Desktop\OneDrive - Arizona State University\Remote PC\NSF Project\nsf-fmrg-data-challenge-main\notebooks\Extraction Ready\sem_multimodal_bundle.py:239: RuntimeWarning: Mean of empty slice
  "y_left_mm": float(np.nanmean(dist["y_left_mm"])),
C:\Users\rolabiyi\OneDrive - Arizona State University\Desktop\OneDrive - Arizona State University\Remote PC\NSF Project\nsf-fmrg-data-challenge-main\notebooks\Extraction Ready\sem_multimodal_bundle.py:240: RuntimeWarning: Mean of empt

In [15]:
from run_feature_ablation import run_feature_ablation

ablation_df = run_feature_ablation(train_df, test_df, ard=False)
print(ablation_df.to_string(index=False))

MAE:  0.1534 mm
NLL:  -0.0245
calibration (nominal vs empirical coverage):
 nominal_coverage  empirical_coverage
             0.50            0.746667
             0.80            0.930000
             0.90            0.980000
             0.95            0.990000
MAE:  0.1319 mm
NLL:  -0.0979
calibration (nominal vs empirical coverage):
 nominal_coverage  empirical_coverage
             0.50            0.783333
             0.80            0.953333
             0.90            0.983333
             0.95            0.996667
MAE:  0.2467 mm
NLL:  0.5632
calibration (nominal vs empirical coverage):
 nominal_coverage  empirical_coverage
             0.50            0.926667
             0.80            0.990000
             0.90            0.993333
             0.95            0.993333
MAE:  0.1095 mm
NLL:  -0.1245
calibration (nominal vs empirical coverage):
 nominal_coverage  empirical_coverage
             0.50            0.863333
             0.80            0.976667
             0.90